In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Load & Preprocess the Data

In [2]:
%%capture
!pip install transformers datasets accelerate -q
!pip install evaluate

In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, 
                          TrainingArguments, EarlyStoppingCallback)
from torch.utils.data import DataLoader, TensorDataset
import evaluate
import os
import gc

In [4]:
# Set CUDA_LAUNCH_BLOCKING for synchronous error reporting
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [5]:
# Load Data
train_df = pd.read_csv('/kaggle/input/hope-speech-detection-across-multiple-languages/Training_Data_PolyHope-M_2025/Training Data/Ur_train.csv')
dev_df = pd.read_csv('/kaggle/input/hope-speech-detection-across-multiple-languages/Training_Data_PolyHope-M_2025/Training Data/Ur_dev.csv')
test_df = pd.read_csv('/kaggle/input/hope-speech-detection-across-multiple-languages/TestWithNoLabel/TestWithNoLabel/Ur_test_without_labels.csv')


In [6]:
# Label Mapping
label_mapping = {'Not Hope': 0, 'Generalized Hope': 1, 'Realistic Hope': 2, 'Unrealistic Hope': 3}
train_df['multiclass'] = train_df['multiclass'].map(label_mapping)
dev_df['multiclass'] = dev_df['multiclass'].map(label_mapping)

# Drop Unnecessary Columns & Handle Missing Data
train_df.drop(columns=['binary'], inplace=True)
dev_df.drop(columns=['binary'], inplace=True)
train_df.dropna(inplace=True)
dev_df.dropna(inplace=True)

In [7]:
# Custom Dataset Class
class HopeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [8]:
# Model and Tokenizer Pairs
model_configs = [
    {"path": "xlm-roberta-base", "name": "XLM-RoBERTa"},
    {"path": "bert-base-multilingual-cased", "name": "BERT-Multilingual"},
    {"path": "distilbert-base-multilingual-cased", "name": "DistilBERT-Multilingual"}
]

In [9]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=3,
    num_train_epochs=36,
    weight_decay=0.1,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
    seed=42,
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [10]:
# Define Evaluation Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_score = f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1": f1_score["f1"]}

In [11]:
# Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Train & Save Models
models = []
for i, config in enumerate(model_configs):
    print(f"Training model {i+1}/{len(model_configs)}: {config['path']}")
    
    # Load Tokenizer and Model
    tokenizer = AutoTokenizer.from_pretrained(config["path"])
    model = AutoModelForSequenceClassification.from_pretrained(config["path"], num_labels=4)
    
    # Update Dropout
    model.config.hidden_dropout_prob = 0.3
    model.config.attention_probs_dropout_prob = 0.3
    
    # Analyze Token Length Distribution (for reference)
    train_text_lengths = [len(tokenizer.tokenize(text)) for text in train_df['text']]
    max_length = min(int(np.percentile(train_text_lengths, 90)), 256)
    print(f"{config['name']} Max Length: {max_length}")
    
    # Tokenize Dataset with Model-Specific Tokenizer
    train_encodings = tokenizer(train_df["text"].tolist(), padding=True, truncation=True, max_length=max_length)
    dev_encodings = tokenizer(dev_df["text"].tolist(), padding=True, truncation=True, max_length=max_length)
    train_dataset = HopeDataset(train_encodings, train_df["multiclass"].tolist())
    dev_dataset = HopeDataset(dev_encodings, dev_df["multiclass"].tolist())
    
    # Move Model to GPU
    model.to(device)
    
    # Train
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
    trainer.train()
    trainer.evaluate()
    
    # Save Model and Tokenizer
    model.save_pretrained(f"/kaggle/working/model_{i}")
    tokenizer.save_pretrained(f"/kaggle/working/model_{i}")
    
    # Store model for ensemble
    models.append(model)
    
    # Clear GPU Memory
    del trainer
    del model
    torch.cuda.empty_cache()
    gc.collect()

Training model 1/3: xlm-roberta-base


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Token indices sequence length is longer than the specified maximum sequence length for this model (1053 > 512). Running this sequence through the model will result in indexing errors


DistilBERT-Multilingual Max Length: 256


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.871500,0.748881,0.676400,0.367877
2,0.728200,0.688070,0.689511,0.379350
3,0.676300,0.639677,0.709178,0.432792
4,0.647700,0.622323,0.716329,0.461415
5,0.561400,0.614635,0.736591,0.499448
6,0.488800,0.651527,0.723480,0.523914
7,0.453400,0.670472,0.722884,0.531834
8,0.340700,0.715537,0.743743,0.581827
9,0.319200,0.807850,0.732420,0.559413
10,0.220200,0.855353,0.731228,0.560705


In [12]:
# Ensemble Prediction on Test Set
test_texts = test_df["text"].tolist()
predictions = []

with torch.no_grad():
    for i, config in enumerate(model_configs):
        # Load saved model and tokenizer
        tokenizer = AutoTokenizer.from_pretrained(f"/kaggle/working/model_{i}")
        model = AutoModelForSequenceClassification.from_pretrained(f"/kaggle/working/model_{i}")
        model.to(device)
        model.eval()
        
        # Tokenize test data with model-specific tokenizer
        encoded_inputs = tokenizer(test_texts, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
        input_ids = encoded_inputs["input_ids"]
        attention_mask = encoded_inputs["attention_mask"]
        dataset = TensorDataset(input_ids, attention_mask)
        dataloader = DataLoader(dataset, batch_size=8)
        
        model_preds = []
        for batch in dataloader:
            input_ids, attention_mask = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            model_preds.append(torch.softmax(logits, dim=-1).cpu())
        
        # Concatenate predictions for this model
        model_preds = torch.cat(model_preds, dim=0)
        predictions.append(model_preds)
        
        # Clear memory
        del model
        torch.cuda.empty_cache()
        gc.collect()

# Average predictions across models
avg_logits = torch.mean(torch.stack(predictions), dim=0)
final_predictions = torch.argmax(avg_logits, dim=1).numpy()

# Map Predictions to Labels
label_map = {0: "Not Hope", 1: "Generalized Hope", 2: "Realistic Hope", 3: "Unrealistic Hope"}
predicted_labels = [label_map[pred] for pred in final_predictions]

# Save Predictions
submission_df = pd.DataFrame({"Text": test_texts, "Tag": predicted_labels})
submission_df.to_csv("predictions.csv", index=False)

print("✅ Predictions saved to 'predictions.csv' using an ensemble model!")

# Final Cleanup
torch.cuda.empty_cache()
gc.collect()

✅ Predictions saved to 'predictions.csv' using an ensemble model!


0